# nets

> Neural networks


In [1]:
#| default_exp nets

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| hide
# from IPython.display import clear_output, DisplayHandle


# def update_patch(self, obj):
#     clear_output(wait=True)
#     self.display(obj)


# DisplayHandle.update = update_patch

In [4]:
from torch import randn as torch_randn
from fastai.vision.all import test_eq

In [5]:
#| export

from fastai.vision.all import ConvLayer, Lambda, MaxPool, NormType, nn, np
from torch import cat as torch_cat
import torch.nn.functional as F
from bioMONAI.core import attributesFromDict

In [6]:
from torch import device as torch_device
from torch.cuda import is_available as cuda_is_available
device = torch_device("cuda" if cuda_is_available() else "cpu")
print(device)

cpu


## DnCNN


In [7]:
#| export

class DnCNN(nn.Module):
    def __init__(self, channels, num_of_layers=9, features=64, kernel_size=3):
        super(DnCNN, self).__init__()
        padding = 1
        layers = []
        layers.append(ConvLayer(channels, features,
                      ks=kernel_size, padding=padding, norm_type=None))
        for _ in range(num_of_layers-2):
            layers.append(ConvLayer(features, features,
                          ks=kernel_size, padding=padding))
        layers.append(nn.Conv2d(in_channels=features, out_channels=channels,
                      kernel_size=kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        residual = self.dncnn(x)
        denoised = x - residual
        return denoised

In [8]:
x = torch_randn(16, 1, 32, 64)

tst = DnCNN(1)
test_eq(tst(x).shape, x.shape)

## UNet


### Convolutional sub-unit


In [9]:
#| export
def SubNetConv(ks=3,
               stride=1,
               padding=None,
               bias=None,
               ndim=2,
               norm_type=NormType.Batch,
               bn_1st=True,
               act_cls=nn.ReLU,
               transpose=False,
               init='auto',
               xtra=None,
               bias_std=0.01,
               dropout=0.0,
               ):

    def _conv(n_in, n_out, n_conv=1):
        s = ConvLayer(n_in, n_out, ks=ks, stride=stride, padding=padding, bias=bias, ndim=ndim, norm_type=norm_type, bn_1st=bn_1st,
                      act_cls=act_cls, transpose=transpose, init=init, xtra=xtra, bias_std=bias_std)
        if dropout is not None and dropout > 0:
            s = nn.Sequential(s, nn.Dropout(dropout))
        for _ in range(n_conv-1):
            t = ConvLayer(n_out, n_out, ks=ks, stride=stride, padding=padding, bias=bias, ndim=ndim, norm_type=norm_type, bn_1st=bn_1st,
                          act_cls=act_cls, transpose=transpose, init=init, xtra=xtra, bias_std=bias_std)
            if dropout is not None and dropout > 0:
                t = nn.Sequential(t, nn.Dropout(dropout))
            s = nn.Sequential(s, t)
        return s

    return _conv

In [10]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

# reduce
tst = SubNetConv(3, padding=1, stride=2, ndim=xdim,
                 norm_type=NormType.Batch, dropout=.1)(1, 2, 2)
y = tst(x)
test_eq(y.shape, [16, 2, 8, 16, 16])
print(tst)
# upsample
tst = SubNetConv(ks=4, padding=0, stride=4, ndim=xdim, norm_type=NormType.Batch,
                 transpose=True)(2, 1)  # to double the size, the kernel cannot be odd
test_eq(tst(y).shape, x.shape)
del y

Sequential(
  (0): Sequential(
    (0): ConvLayer(
      (0): Conv3d(1, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
  (1): Sequential(
    (0): ConvLayer(
      (0): Conv3d(2, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
)


### UNet block


In [11]:
#| export
class _Net_recurse(nn.Module):
    def __init__(self,
                 depth=4,						# depth of the UNet network
                 mult_chan=32,					# number of filters at first layer
                 in_channels=1,					# number of input channels
                 kernel_size=3,					# kernel size of convolutional layers
                 ndim=2,							# number of spatial dimensions of the input data
                 n_conv_per_depth=2,				# number of convolutions per layer
                 activation=nn.ReLU,				# activation function used in convolutional layers
                 norm_type=NormType.Batch,
                 dropout=0.0,
                 pool=MaxPool,
                 pool_size=2,
                 ):
        """Class for recursive definition of U-network.p

        Parameters:
        in_channels - (int) number of channels for input.
        mult_chan - (int) factor to determine number of output channels
        depth - (int) if 0, this subnet will only be convolutions that double the channel count.
        """
        super().__init__()
        # Parameters
        self.depth = depth
        n_out = in_channels*mult_chan

        # Layer types
        Pooling = pool(ks=pool_size, ndim=ndim)
        UpSample = nn.Upsample(scale_factor=pool_size, mode='nearest')
        SubNet_Conv = SubNetConv(ks=kernel_size, stride=1, padding=None, bias=None, ndim=ndim, norm_type=norm_type,
                                 bn_1st=True, act_cls=activation, transpose=False, dropout=dropout)

        # Blocks
        self.sub_conv_more = SubNet_Conv(in_channels, n_out, n_conv_per_depth)
        if self.depth > 0:
            in_channels = n_out
            mult_chan = 2
            depth = (self.depth - 1)
            self.sub_u = nn.Sequential(Pooling,                                                         # layer reducing the image size (usually a pooling layer)
                                       _Net_recurse(depth, mult_chan, in_channels, kernel_size,
                                                    ndim, n_conv_per_depth, activation, norm_type,
                                                    dropout, pool, pool_size),                          # lower unet level
                                       # layer increasing the image size (usually an upsampling layer)
                                       UpSample,
                                       )
            self.sub_conv_less = SubNet_Conv(3*n_out, n_out, n_conv_per_depth)

    def forward(self, x):
        if self.depth == 0:
            return self.sub_conv_more(x)
        else:  # depth > 0
            # convolutions with increasing number of channels
            x_conv_more = self.sub_conv_more(x)
            x_from_sub_u = self.sub_u(x_conv_more)
            # concatenate the upsampled outputs of the lower level with the outputs of the next level in size
            x_cat = torch_cat((x_from_sub_u, x_conv_more), 1)
            # convolutions with decreasing number of channels
            x_conv_less = self.sub_conv_less(x_cat)
        return x_conv_less

### Recursive UNet


In [12]:
#| export
class UNet(nn.Module):
    def __init__(self,
                 depth=4,						# depth of the UNet network
                 mult_chan=32,					# number of filters at first layer
                 in_channels=1,					# number of input channels
                 out_channels=1,					# number of output channels
                 last_activation=None,			# last activation before final result
                 kernel_size=3,					# kernel size of convolutional layers
                 ndim=2,							# number of spatial dimensions of the input data
                 n_conv_per_depth=2,				# number of convolutions per layer
                 activation='ReLU',				# activation function used in convolutional layers
                 norm_type=NormType.Batch,
                 dropout=0.0,
                 pool=MaxPool,
                 pool_size=2,
                 residual=False,
                 prob_out=False,
                 eps_scale=1e-3,
                 ):
        super().__init__()
        last_activation = getattr(F, f"{activation.lower()}") if last_activation == None else getattr(
            F, f"{last_activation.lower()}")
        activation = getattr(nn, f"{activation}")
        attributesFromDict(locals())		# stores all the input parameters in self

        self.net_recurse = _Net_recurse(depth, mult_chan, in_channels, kernel_size, ndim,
                                        n_conv_per_depth, activation, norm_type, dropout, pool, pool_size)
        self.conv_out = ConvLayer(mult_chan*in_channels, out_channels, ndim=ndim,
                                  ks=kernel_size, norm_type=None, act_cls=None, padding=1)

    def forward(self, x):
        x_rec = self.net_recurse(x)
        final = self.conv_out(x_rec)

        if self.residual:
            if not (self.out_channels == self.in_channels):
                raise ValueError(
                    "number of input and output channels must be the same for a residual net.")
            final = final + x
        final = self.last_activation(final)

        if self.prob_out:
            scale = ConvLayer(self.out_channels, self.out_channels,
                              ndim=self.ndim, ks=1, norm_type=None, act_cls=nn.Softplus)(x_rec)
            scale = Lambda(lambda x: x+np.float32(self.eps_scale))(scale)
            final = torch_cat((final, scale), 1)

        return final

In [13]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

tst = UNet(depth=1, ndim=xdim, n_conv_per_depth=1, residual=True)
mods = list(tst.children())
print(mods)
test_eq(tst(x).shape, [16, 1, 32, 64, 64])

[_Net_recurse(
  (sub_conv_more): ConvLayer(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (sub_u): Sequential(
    (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (1): _Net_recurse(
      (sub_conv_more): ConvLayer(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (2): Upsample(scale_factor=2.0, mode='nearest')
  )
  (sub_conv_less): ConvLayer(
    (0): Conv3d(96, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
), ConvLayer(
  (0): Conv3d(32, 1, kernel_size=(3, 3, 3), stride=(1, 

## DeepLab v3

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.blocks import Convolution
from monai.networks.layers.factories import Act, Norm, Pool
from torchvision.models.resnet import resnet50, resnet101
from monai.networks.nets import resnet


### Config

In [ ]:

@dataclass
class DeeplabConfig:
    dimensions: int
    in_channels: int
    out_channels: int
    backbone: str = "xception" 
    pretrained: bool = False
    middle_flow_blocks: int = 16
    aspp_dilations: List[int] = field(default_factory=lambda: [1, 6, 12, 18])
    entry_block3_stride: int = 2
    middle_block_dilation: int = 1
    exit_block_dilations: Tuple[int, int] = (1, 2)

def get_padding(kernel_size: int, dilation: int) -> int:
    return (kernel_size - 1) * dilation // 2

def interpolate(x: torch.Tensor, size: Union[List[int], Tuple[int, ...]], mode: str) -> torch.Tensor:
    if x.dim() == 4:  # 2D
        return F.interpolate(x, size=size, mode=mode, align_corners=True)
    elif x.dim() == 5:  # 3D
        return F.interpolate(x, size=size, mode=mode, align_corners=True)
    else:
        raise ValueError(f"Unsupported input dimension: {x.dim()}")

### Blocks

In [ ]:
class SeparableConv(nn.Module):
    def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, kernel_size: int = 3, stride: int = 1, 
                 dilation: int = 1, bias: bool = False, norm: Optional[str] = None):
        super().__init__()
        self.conv1 = Convolution(
            dimensions=config.dimensions, 
            in_channels=inplanes, 
            out_channels=inplanes, 
            kernel_size=kernel_size,
            groups=inplanes, 
            padding=get_padding(kernel_size, dilation), 
            dilation=dilation, 
            bias=bias, 
            strides=stride
        )
        self.pointwise = Convolution(
            dimensions=config.dimensions, 
            in_channels=inplanes, 
            out_channels=planes, 
            kernel_size=1, 
            strides=1,
            padding=0, 
            dilation=1, 
            groups=1, 
            bias=bias,
            norm=Norm.BATCH if norm else None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.pointwise(x)
        return x

class Block(nn.Module):
    def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, reps: int, stride: int = 1, 
                 dilation: int = 1, start_with_relu: bool = True, grow_first: bool = True, 
                 is_last: bool = False):
        super().__init__()
        if planes != inplanes or stride != 1:
            self.skip = Convolution(config.dimensions, inplanes, planes, kernel_size=1, bias=False, 
                                    strides=stride, norm=Norm.BATCH)
        else:
            self.skip = None

        self.relu = nn.ReLU(inplace=True)
        rep = []

        filters = inplanes
        if grow_first:
            rep.append(self.relu)
            rep.append(SeparableConv(config, inplanes, planes, 3, stride=1, dilation=dilation, norm=Norm.BATCH))
            filters = planes

        for _ in range(reps - 1):
            rep.append(self.relu)
            rep.append(SeparableConv(config, filters, filters, 3, stride=1, dilation=dilation, norm=Norm.BATCH))

        if not grow_first:
            rep.append(self.relu)
            rep.append(SeparableConv(config, inplanes, planes, 3, stride=1, dilation=dilation, norm=Norm.BATCH))

        if not start_with_relu:
            rep = rep[1:]

        if stride != 1:
            rep.append(SeparableConv(config, planes, planes, 3, stride=2))

        if stride == 1 and is_last:
            rep.append(SeparableConv(config, planes, planes, 3, stride=1))

        self.rep = nn.Sequential(*rep)

    def forward(self, inp: torch.Tensor) -> torch.Tensor:
        x = self.rep(inp)
        if self.skip is not None:
            skip = self.skip(inp)
        else:
            skip = inp
        x += skip
        return x

### Aligned Xception

In [ ]:
class Xception(nn.Module):
    def __init__(self, config: DeeplabConfig):
        super().__init__()
        self.config = config

        self.conv1 = Convolution(config.dimensions, config.in_channels, 32, kernel_size=3,
                                 bias=False, strides=2, padding=1, norm=Norm.BATCH)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = Convolution(config.dimensions, 32, 64, kernel_size=3,
                                 bias=False, strides=1, padding=1, norm=Norm.BATCH)

        self.block1 = Block(config, 64, 128, reps=2, stride=2, start_with_relu=False)
        self.block2 = Block(config, 128, 256, reps=2, stride=2, start_with_relu=True, grow_first=True)
        self.block3 = Block(config, 256, 728, reps=2, stride=config.entry_block3_stride, 
                            start_with_relu=True, grow_first=True, is_last=True)

        # Middle flow
        self.middle_flow = nn.Sequential(*[
            Block(config, 728, 728, reps=3, stride=1, dilation=config.middle_block_dilation,
                  start_with_relu=True, grow_first=True)
            for _ in range(config.middle_flow_blocks)
        ])

        # Exit flow
        self.exit_block = Block(config, 728, 1024, reps=2, stride=1, dilation=config.exit_block_dilations[0],
                             start_with_relu=True, grow_first=False, is_last=True)

        self.conv3 = SeparableConv(config, 1024, 1536, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)
        self.conv4 = SeparableConv(config, 1536, 1536, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)
        self.conv5 = SeparableConv(config, 1536, 2048, 3, stride=1, dilation=config.exit_block_dilations[1], norm=Norm.BATCH)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Entry flow
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)

        x = self.block1(x)
        low_level_feat = x
        x = self.block2(x)
        x = self.block3(x)

        # Middle flow
        x = self.middle_flow(x)

        # Exit flow
        x = self.exit_block(x)
        x = self.conv3(x)
        x = self.relu(x)
        x = self.conv4(x)
        x = self.relu(x)
        x = self.conv5(x)
        x = self.relu(x)

        return x, low_level_feat

### ASPP

In [ ]:
class ASPP_module(nn.Module):
    def __init__(self, config: DeeplabConfig, inplanes: int, planes: int, dilation: int):
        super().__init__()
        kernel_size = 1 if dilation == 1 else 3
        padding = 0 if dilation == 1 else dilation
        self.atrous_convolution = Convolution(config.dimensions, inplanes, planes, kernel_size=kernel_size,
                                              strides=1, padding=padding, dilation=dilation, 
                                              bias=False, norm=Norm.BATCH)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.atrous_convolution(x)
        return self.relu(x)

### DeepLab

In [ ]:
class Deeplab(nn.Module):
    def __init__(self, config: DeeplabConfig):
        super().__init__()
        self.config = config

        # Choose backbone based on configuration
        if config.backbone == "xception":
            self.backbone = Xception(config)
            backbone_out_channels = 2048
        elif config.backbone == "resnet50":
            self.backbone = resnet.resnet50(pretrained=config.pretrained, spatial_dims=config.dimensions)
            del self.backbone.fc
            del self.backbone.avgpool
            backbone_out_channels = 2048
        elif config.backbone == "resnet101":
            self.backbone = resnet.resnet101(pretrained=config.pretrained, spatial_dims=config.dimensions)
            del self.backbone.fc
            del self.backbone.avgpool
            backbone_out_channels = 2048
        else:
            raise ValueError(f"Unsupported backbone: {config.backbone}")

        # ASPP
        self.aspp_modules = nn.ModuleList([
            ASPP_module(config, backbone_out_channels, 256, dilation=dilation)
            for dilation in config.aspp_dilations
        ])

        self.global_avg_pool = nn.Sequential(
            Pool[Pool.ADAPTIVEAVG, config.dimensions](1),
            Convolution(config.dimensions, backbone_out_channels, 256, kernel_size=1, strides=1, bias=False, norm=Norm.BATCH),
            nn.ReLU()
        )

        self.conv1 = Convolution(config.dimensions, 1280, 256, kernel_size=1, bias=False, norm=Norm.BATCH)
        self.conv2 = Convolution(config.dimensions, 256, 48, kernel_size=1, bias=False, norm=Norm.BATCH)

        self.last_conv = nn.Sequential(
            Convolution(config.dimensions, 304, 256, kernel_size=3, strides=1, padding=1, bias=False, norm=Norm.BATCH),
            nn.ReLU(),
            Convolution(config.dimensions, 256, 256, kernel_size=3, strides=1, padding=1, bias=False, norm=Norm.BATCH),
            nn.ReLU(),
            Convolution(config.dimensions, 256, config.out_channels, kernel_size=1, strides=1)
        )

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        if self.config.backbone == "xception":
            x, low_level_features = self.backbone(input)
        else:
            x = self.backbone.conv1(input)
            x = self.backbone.bn1(x)
            x = self.backbone.relu(x)
            x = self.backbone.maxpool(x)

            low_level_features = self.backbone.layer1(x)
            x = self.backbone.layer2(low_level_features)
            x = self.backbone.layer3(x)
            x = self.backbone.layer4(x)

        aspp_results = [module(x) for module in self.aspp_modules]
        x5 = self.global_avg_pool(x)
        x5 = interpolate(x5, size=x.shape[2:], mode='linear' if self.config.dimensions == 2 else 'trilinear')
        x = torch.cat(aspp_results + [x5], dim=1)

        x = self.conv1(x)
        x = interpolate(x, size=low_level_features.shape[2:], mode='linear' if self.config.dimensions == 2 else 'trilinear')

        low_level_features = self.conv2(low_level_features)

        x = torch.cat((x, low_level_features), dim=1)
        x = self.last_conv(x)
        x = interpolate(x, size=input.shape[2:], mode='linear' if self.config.dimensions == 2 else 'trilinear')

        return x

#### Example

In [ ]:
# For 2D images
config_2d = DeeplabConfig(
    dimensions=2,
    in_channels=3,  # For RGB images
    out_channels=4,
    middle_flow_blocks=16,
    aspp_dilations=[1, 6, 12, 18]
)
model_2d = Deeplab(config_2d)

# For 3D images
config_3d = DeeplabConfig(
    dimensions=3,
    in_channels=1,  # For single-channel 3D medical images
    out_channels=4,
    middle_flow_blocks=16,
    aspp_dilations=[1, 6, 12, 18]
)
model_3d = Deeplab(config_3d)

---


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()